In [1]:
from module import *

In [2]:
# Explore database schema
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)
print("Tables in database:")
print(tables['name'].tolist())
print("\n" + "="*60 + "\n")

# Display schema for each table
for table in tables['name']:
    print(f"TABLE: {table}")
    schema = pd.read_sql(f"PRAGMA table_info({table});", conn)
    print(schema[['name', 'type', 'notnull']])
    
    # Show sample data
    sample = pd.read_sql(f"SELECT * FROM {table} LIMIT 3;", conn)
    print(f"Sample ({len(sample)} rows):")
    print(sample.to_string())
    print("\n" + "="*60 + "\n")

Tables in database:
['daily_activity', 'heart_rate', 'hourly_calories', 'hourly_intensity', 'hourly_steps', 'minute_sleep', 'weight_log']


TABLE: daily_activity
                        name     type  notnull
0                         Id     REAL        0
1               ActivityDate     TEXT        0
2                 TotalSteps  INTEGER        0
3              TotalDistance     REAL        0
4            TrackerDistance     REAL        0
5   LoggedActivitiesDistance     REAL        0
6         VeryActiveDistance     REAL        0
7   ModeratelyActiveDistance     REAL        0
8        LightActiveDistance     REAL        0
9    SedentaryActiveDistance     REAL        0
10         VeryActiveMinutes  INTEGER        0
11       FairlyActiveMinutes  INTEGER        0
12      LightlyActiveMinutes  INTEGER        0
13          SedentaryMinutes  INTEGER        0
14                  Calories  INTEGER        0
Sample (3 rows):
             Id ActivityDate  TotalSteps  TotalDistance  TrackerDista

In [3]:
# List all tables
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)
table_list = tables['name'].tolist()
print("Tables in database:")
for table in table_list:
    print(f"  - {table}")

Tables in database:
  - daily_activity
  - heart_rate
  - hourly_calories
  - hourly_intensity
  - hourly_steps
  - minute_sleep
  - weight_log


In [4]:
# Show schema details and row counts for each table
for table in table_list:
    count = pd.read_sql(f"SELECT COUNT(*) as count FROM {table};", conn)['count'][0]
    schema = pd.read_sql(f"PRAGMA table_info({table});", conn)
    
    print(f"\n{table} ({count:,} rows)")
    print("-" * 50)
    for _, row in schema.iterrows():
        print(f"  {row['name']:20} {row['type']:15}")
    
    # Show first row
    sample = pd.read_sql(f"SELECT * FROM {table} LIMIT 1;", conn)
    if not sample.empty:
        print("\nSample row:")
        for col in sample.columns:
            print(f"  {col}: {sample[col].iloc[0]}")


daily_activity (457 rows)
--------------------------------------------------
  Id                   REAL           
  ActivityDate         TEXT           
  TotalSteps           INTEGER        
  TotalDistance        REAL           
  TrackerDistance      REAL           
  LoggedActivitiesDistance REAL           
  VeryActiveDistance   REAL           
  ModeratelyActiveDistance REAL           
  LightActiveDistance  REAL           
  SedentaryActiveDistance REAL           
  VeryActiveMinutes    INTEGER        
  FairlyActiveMinutes  INTEGER        
  LightlyActiveMinutes INTEGER        
  SedentaryMinutes     INTEGER        
  Calories             INTEGER        

Sample row:
  Id: 1503960366.0
  ActivityDate: 3/25/2016
  TotalSteps: 11004
  TotalDistance: 7.1100001335144
  TrackerDistance: 7.1100001335144
  LoggedActivitiesDistance: 0.0
  VeryActiveDistance: 2.5699999332428
  ModeratelyActiveDistance: 0.46000000834465
  LightActiveDistance: 4.07000017166138
  SedentaryActiveDistance

In [5]:
# Quick summary of data size and columns
summary = []
for table in table_list:
    count = pd.read_sql(f"SELECT COUNT(*) as count FROM {table};", conn)['count'][0]
    schema = pd.read_sql(f"PRAGMA table_info({table});", conn)
    col_count = len(schema)
    columns = ", ".join(schema['name'].tolist()[:3]) + ("..." if col_count > 3 else "")
    summary.append({"Table": table, "Rows": f"{count:,}", "Columns": col_count, "Sample Cols": columns})

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

           Table      Rows  Columns                         Sample Cols
  daily_activity       457       15     Id, ActivityDate, TotalSteps...
      heart_rate 1,154,681        3                     Id, Time, Value
 hourly_calories    24,084        3          Id, ActivityHour, Calories
hourly_intensity    24,084        4 Id, ActivityHour, TotalIntensity...
    hourly_steps    24,084        3         Id, ActivityHour, StepTotal
    minute_sleep   198,559        4                  Id, date, value...
      weight_log        33        8               Id, Date, WeightKg...


In [6]:
# Check date ranges for each dataset
print("\nDate ranges:")
print("-" * 50)

# Daily activity
daily = pd.read_sql("SELECT MIN(ActivityDate) as min_date, MAX(ActivityDate) as max_date FROM daily_activity;", conn)
print(f"Daily Activity: {daily['min_date'][0]} to {daily['max_date'][0]}")

# Heart rate
hr = pd.read_sql("SELECT MIN(Time) as min_date, MAX(Time) as max_date FROM heart_rate;", conn)
print(f"Heart Rate: {hr['min_date'][0][:10]} to {hr['max_date'][0][:10]}")

# Sleep
sleep = pd.read_sql("SELECT MIN(date) as min_date, MAX(date) as max_date FROM minute_sleep;", conn)
print(f"Sleep: {sleep['min_date'][0][:10]} to {sleep['max_date'][0][:10]}")

# Weight
weight = pd.read_sql("SELECT MIN(Date) as min_date, MAX(Date) as max_date FROM weight_log;", conn)
print(f"Weight: {weight['min_date'][0][:10]} to {weight['max_date'][0][:10]}")


Date ranges:
--------------------------------------------------
Daily Activity: 3/12/2016 to 4/9/2016
Heart Rate: 3/29/2016  to 4/9/2016 9
Sleep: 3/11/2016  to 4/9/2016 9
Weight: 3/30/2016  to 4/9/2016 8
